# 🚀 Huấn Luyện Mô Hình Deep Learning Sửa Lỗi Chính Tả Tiếng Việt (ViT5 Transformer)
**Môn học**: Học sâu (3 tín chỉ) - Đồ án: Phát hiện & Tự động sửa lỗi chính tả tiếng Việt

> **Lưu ý**: Hãy vào menu **Runtime > Change runtime type** và chọn **T4 GPU** để huấn luyện siêu tốc (10-15 phút) thay vì dùng CPU mất nhiều ngày!

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi

In [ ]:
# 2. Cài đặt thư viện chuyên dụng
!pip install -q transformers datasets accelerate sentencepiece evaluate jiwer rapidfuzz

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset
import evaluate
import numpy as np
import random

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Đang huấn luyện trên thiết bị: {device}")

In [ ]:
# 3. Tải mô hình ViT5 và Tokenizer từ HuggingFace
MODEL_NAME = "VietAI/vit5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
print("[*] Đã nạp thành công mô hình ViT5!")

In [ ]:
# 4. Chuẩn bị ngữ liệu huấn luyện (Đa dạng các loại lỗi: telex, vni, phím lân cận, thiếu dấu)
SAMPLE_DATA = [
    ("sở thích của tôu là học tiếng anh", "sở thích của tôi là học tiếng Anh"),
    ("Tối nay6 tôi tự nấu ăn ở nhà để tiet kiệm fiền", "Tối nay tôi tự nấu ăn ở nhà để tiết kiệm tiền"),
    ("Hôm nayw tôi không đi họt đi trời jưa", "Hôm nay tôi không đi học vì trời mưa"),
    ("Mô hình uọc sâu cà ma5ng nơ-ronj biên đổi cách fon người giải quyết bàu toán phức tạo.", "Mô hình học sâu và mạng nơ-ron biến đổi cách con người giải quyết bài toán phức tạp."),
    ("Trường dạy học Bách Khoa co về dày lịch sử và truyền thongw vẽ vang.", "Trường đại học Bách Khoa có bề dày lịch sử và truyền thống vẻ vang."),
    ("Tôi dất thik đocj truyện tranh vào buỏi chiều tối", "Tôi rất thích đọc truyện tranh vào buổi chiều tối"),
    ("Chời hôm nay dất đẹp và trong xabh", "Trời hôm nay rất đẹp và trong xanh"),
    ("Thời tiếc miền bắc mùa đông xe lạnh và có mưa fùn nhẹ.", "Thời tiết miền Bắc mùa đông se lạnh và có mưa phùn nhẹ."),
    ("Học phần học sâu giúp sinh viên nắm vũng kiến thức mạng nơ ron", "Học phần học sâu giúp sinh viên nắm vững kiến thức mạng nơ-ron"),
    ("Việt Nam là một đất nc hoà bình và phát chiển", "Việt Nam là một đất nước hoà bình và phát triển"),
]

# Nhân bản dữ liệu để học sâu hội tụ chuẩn xác
augmented_train = []
for _ in range(200):
    for src, tgt in SAMPLE_DATA:
        augmented_train.append({"input_text": f"sửa lỗi: {src}", "target_text": tgt})

train_dataset = Dataset.from_list(augmented_train)
print(f"[*] Số mẫu huấn luyện: {len(train_dataset)}")

In [ ]:
# 5. Tiền xử lý Tokenize
def preprocess_function(examples):
    inputs = [ex for ex in examples["input_text"]]
    targets = [ex for ex in examples["target_text"]]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(preprocess_function, batched=True)

In [ ]:
# 6. Thiết lập tham số huấn luyện chuẩn cho GPU
training_args = Seq2SeqTrainingArguments(
    output_dir="./vit5_checkpoints",
    evaluation_strategy="no",
    learning_rate=1e-4,
    per_device_train_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    tokenizer=tokenizer,
)

print("[*] Bắt đầu huấn luyện ViT5 trên GPU...")
trainer.train()
print("[*] Huấn luyện hoàn tất!")

In [ ]:
# 7. Xuất mô hình tốt nhất về thư mục best_model
save_path = "./best_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"[*] Đã lưu mô hình tốt nhất vào: {save_path}")

# Nén file để tải về máy tính:
!zip -r vit5_best_model.zip best_model
print("[*] Đã nén thành vit5_best_model.zip. Hãy tải file này về và giải nén vào checkpoints/vit5_spelling_corrector/best_model!")